Veri Birleştirme

In [5]:
import pandas as pd
import numpy as np

# 1. Ham verileri oku
df_teslim = pd.read_excel("../data/raw/dogo_teslim_edilenler.xlsx")
df_iptal = pd.read_excel("../data/raw/dogo_iptal.xlsx")
df_iade = pd.read_excel("../data/raw/dogo_iade.xlsx")
df_iade_detay = pd.read_excel("../data/raw/dogo_iade_aciklamali.xlsx")

# 2. Sipariş Numaralarını temizle (string ve boşluksuz)
for df in [df_teslim, df_iptal, df_iade, df_iade_detay]:
    if 'Sipariş No' in df.columns:
        df['Sipariş No'] = df['Sipariş No'].astype(str).str.strip()

# 3. Durum etiketlerini ata
df_teslim['durum'] = 'teslim'
df_iptal['durum'] = 'iptal'
df_iade['durum'] = 'iade'

# 4. Ana siparişleri birleştir
df_siparisler = pd.concat([df_teslim, df_iptal, df_iade], ignore_index=True)

# 5. İade detayındaki verileri Sipariş No bazında soldan birleştir
detay_kolonlar = ['Sipariş No', 'Ürün Adı', 'Alt Ürün Adı', 'İade Nedeni', 'Üye Açıklaması', 'Genel İade Açıklaması']
mevcut_detay_kolonlar = [c for c in detay_kolonlar if c in df_iade_detay.columns]

df_iade_detay_tekil = df_iade_detay[mevcut_detay_kolonlar].drop_duplicates(subset=['Sipariş No'], keep='first')

df_ana = pd.merge(
    df_siparisler, 
    df_iade_detay_tekil, 
    on='Sipariş No', 
    how='left'
)

# 6. KRİTİK GÜNCELLEME: İade açıklaması / talebi olan siparişlerin durumunu 'iade' yap
iade_talebi_olanlar = df_ana['İade Nedeni'].notna() | df_ana['Üye Açıklaması'].notna() | df_ana['Genel İade Açıklaması'].notna()
df_ana.loc[iade_talebi_olanlar, 'durum'] = 'iade'

print("✅ Güncelleme Tamamlandı!")
print(df_ana['durum'].value_counts())

✅ Güncelleme Tamamlandı!
durum
teslim    2576
iade       328
iptal      270
Name: count, dtype: int64


Veri temizliği

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re
import unicodedata

# ---------------------------------------------------------
# 1. YARDIMCI TEMİZLİK FONKSİYONLARI
# ---------------------------------------------------------

def tutar_temizle(s):
    if pd.isna(s):
        return 0.0
    if isinstance(s, (int, float)):
        return float(s)
    s_temiz = re.sub(r'[^\d,-]', '', str(s)).replace(',', '.')
    return float(s_temiz) if s_temiz else 0.0

def telefon_hashle(tel):
    if pd.isna(tel):
        return None
    rakamlar = re.sub(r'\D', '', str(tel))
    return hashlib.sha256(rakamlar.encode()).hexdigest()[:12] if rakamlar else None

def metin_temizle(s):
    """Türkçe karakterleri ve büyük/küçük harf uyumsuzluklarını standartlaştırır."""
    if pd.isna(s) or str(s).strip() in ("", "nan", "NaN"):
        return np.nan
    s = str(s).strip().upper()
    s = s.replace("İ", "I").replace("Ğ", "G").replace("Ü", "U").replace("Ş", "S").replace("Ö", "O").replace("Ç", "C")
    return s

# ---------------------------------------------------------
# 2. VERİ TİPİ VE METİN DÜZELTMELERİ
# ---------------------------------------------------------

# Sayısal alanlar
df_ana['tutar_clean'] = df_ana['Tutar'].apply(tutar_temizle)
if 'KDV ' in df_ana.columns:
    df_ana['kdv_clean'] = df_ana['KDV '].apply(tutar_temizle)
if 'Geçen Süre (dk)' in df_ana.columns:
    df_ana['gecen_sure_dk'] = pd.to_numeric(df_ana['Geçen Süre (dk)'], errors='coerce')

# Maskelenmiş Müşteri ID
tel_kolonu = 'Cep Telefonu (Üye)' if 'Cep Telefonu (Üye)' in df_ana.columns else 'Telefon Numarası'
df_ana['musteri_id'] = df_ana[tel_kolonu].apply(telefon_hashle)

# Tarih ve Zaman Özellikleri
df_ana['tarih_clean'] = pd.to_datetime(df_ana['Tarih'], dayfirst=True, errors='coerce')
df_ana['siparis_ayi'] = df_ana['tarih_clean'].dt.to_period('M').astype(str)
df_ana['siparis_gunu'] = df_ana['tarih_clean'].dt.day_name()
df_ana['siparis_saati'] = df_ana['tarih_clean'].dt.hour

# Metin Normalizasyonları (İl, İlçe, Platform vb.)
for col in ['İl (Teslimat)', 'İlçe (Teslimat)', 'Platform', 'Kaynak', 'Ödeme Tipi', 'Kargo']:
    if col in df_ana.columns:
        df_ana[col + '_clean'] = df_ana[col].apply(metin_temizle)

# Kampanya Parçalama
if 'Kampanya' in df_ana.columns:
    df_ana['kampanyali_mi'] = df_ana['Kampanya'].notna() & (df_ana['Kampanya'].astype(str).str.strip() != "")
    df_ana['kampanya_sayisi'] = df_ana['Kampanya'].fillna("").astype(str).apply(lambda x: len([k for k in x.split("|") if k.strip()]))

# ---------------------------------------------------------
# 3. GEREKSİZ / ÇÖP SÜTUNLARI ELEMEK (ÇEKİRDEK VERİ SETİ)
# ---------------------------------------------------------

# Analizde tutmak istediğimiz temiz kolon listesi
tutulacak_kolonlar = [
    'Sipariş No', 'durum', 'musteri_id', 'tarih_clean', 'siparis_ayi', 'siparis_gunu', 'siparis_saati',
    'tutar_clean', 'gecen_sure_dk',
    'İl (Teslimat)_clean', 'İlçe (Teslimat)_clean', 
    'Platform_clean', 'Kaynak_clean', 'Ödeme Tipi_clean', 'Kargo_clean',
    'kampanyali_mi', 'kampanya_sayisi',
    'Ürün Adı', 'Alt Ürün Adı', 'İade Nedeni', 'Üye Açıklaması', 'Genel İade Açıklaması'
]

# Sadece mevcut olan kolonları filtrele
mevcut_kolonlar = [c for c in tutulacak_kolonlar if c in df_ana.columns]
df_temiz = df_ana[mevcut_kolonlar].copy()

# Kontrol Çıktıları
#print("✅ Detaylı Veri Temizliği Tamamlandı!")
#print(f"Ham Sütun Sayısı: {df_ana.shape[1]} ──> Temizlenmiş Sütun Sayısı: {df_temiz.shape[1]}")

#df_temiz.head()

In [7]:
import pandas as pd

print("🧹 Veri Temizleme: Son Aşama Raporu\n" + "-"*50)

# 1. Eksik Verilerin (NaN / Null) Yönetimi
# Kategorik sütunlardaki boşlukları silmek yerine "BILINMIYOR" etiketiyle dolduruyoruz.
kategorik_kolonlar = [
    'İl (Teslimat)_clean', 'İlçe (Teslimat)_clean', 
    'Platform_clean', 'Kaynak_clean', 'Ödeme Tipi_clean', 'Kargo_clean'
]
mevcut_kategorik = [c for c in kategorik_kolonlar if c in df_temiz.columns]

eksik_oncesi = df_temiz[mevcut_kategorik].isna().sum().sum()
df_temiz.loc[:, mevcut_kategorik] = df_temiz[mevcut_kategorik].fillna("BILINMIYOR")
print(f"✔️ Adım 1: Toplam {eksik_oncesi} adet eksik kategorik hücre 'BILINMIYOR' ile dolduruldu.")

# 2. Metin Standartlaştırma
# Bir önceki hücrede metin_temizle() fonksiyonu bunu yapmıştı, burada ek olarak baştaki/sondaki gizli boşlukları tıraşlıyoruz.
for col in mevcut_kategorik:
    df_temiz.loc[:, col] = df_temiz[col].astype(str).str.strip()
print(f"✔️ Adım 2: Metin standartlaştırma ve gizli boşluk temizliği tamamlandı.")

# 3. Aykırı Değer (Outlier) ve Mantık Kontrolü
# Tutar: 0'dan küçük eksi değerleri veya 500.000 TL'den büyük hatalı girişleri siliyoruz.
satir_sayisi_oncesi = len(df_temiz)
df_temiz = df_temiz[(df_temiz['tutar_clean'] >= 0) & (df_temiz['tutar_clean'] <= 500000)]

# Tarih: 2015'ten öncesine veya bugünden sonrasına ait mantıksız tarihleri siliyoruz.
# (Tarihi boş olanları kaybetmemek için onlara dokunmuyoruz)
bugun = pd.Timestamp.now()
df_temiz = df_temiz[
    (df_temiz['tarih_clean'].isna()) | 
    ((df_temiz['tarih_clean'] >= pd.Timestamp('2015-01-01')) & (df_temiz['tarih_clean'] <= bugun))
]
aykiri_silinen = satir_sayisi_oncesi - len(df_temiz)
print(f"✔️ Adım 3: Mantık dışı tutar veya tarihe sahip {aykiri_silinen} adet aykırı satır (outlier) temizlendi.")

# 4. Mükerrer (Duplicate) Satır Kontrolü
# Tamamen aynı olan (birebir kopyalanmış) hayalet satırları siliyoruz.
satir_sayisi_kopyasiz = len(df_temiz)
df_temiz = df_temiz.drop_duplicates()
kopya_silinen = satir_sayisi_kopyasiz - len(df_temiz)
print(f"✔️ Adım 4: Dışa aktarım hatası olan {kopya_silinen} adet mükerrer (kopya) satır silindi.")

print("-" * 50)
print(f"🚀 NİHAİ TEMİZ TABLO HAZIR! Toplam Kalan Net Satır: {len(df_temiz)}")

🧹 Veri Temizleme: Son Aşama Raporu
--------------------------------------------------
✔️ Adım 1: Toplam 1844 adet eksik kategorik hücre 'BILINMIYOR' ile dolduruldu.
✔️ Adım 2: Metin standartlaştırma ve gizli boşluk temizliği tamamlandı.
✔️ Adım 3: Mantık dışı tutar veya tarihe sahip 0 adet aykırı satır (outlier) temizlendi.
✔️ Adım 4: Dışa aktarım hatası olan 0 adet mükerrer (kopya) satır silindi.
--------------------------------------------------
🚀 NİHAİ TEMİZ TABLO HAZIR! Toplam Kalan Net Satır: 3174


In [8]:
# Temizlenmiş ve son kontrolleri yapılmış veriyi kaydet
df_temiz.to_csv("../data/processed/dogo_temiz_veri.csv", index=False)
print("✅ Temiz veri başarıyla 'data/processed/dogo_temiz_veri.csv' konumuna kaydedildi!")

✅ Temiz veri başarıyla 'data/processed/dogo_temiz_veri.csv' konumuna kaydedildi!
